# Clustering Using Spherical K-means

## Set up Environment

### Install Dependencies

In [1]:
!pip install --upgrade numpy pandas matplotlib wordcloud scikit-learn tqdm

In [2]:
!pip install --upgrade faiss-cpu

### Import Dependencies

In [46]:
import math
import os
import re
import faiss
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from wordcloud import WordCloud
from sklearn.metrics import silhouette_score
from tqdm import tqdm

### Load Data

In [54]:
df = pd.read_pickle('datasets/tickets.pkl')

X_embeddings = np.vstack(df['EMBEDDING'].values)
embedding_dimension = X_embeddings.shape[1]

print("Vektor embedding berhasil dimuat kembali.")
print(f"Jumlah data: {X_embeddings.shape[0]}")
print(f"Dimensi embedding: {embedding_dimension}")

Vektor embedding berhasil dimuat kembali.
Jumlah data: 1639
Dimensi embedding: 256


## Experimenting

### Normalize Data

In [55]:
X_embeddings = np.ascontiguousarray(X_embeddings, dtype=np.float32)
faiss.normalize_L2(X_embeddings)

print("Vektor embedding berhasil dinormalisasi.")

Vektor embedding berhasil dinormalisasi.


### Initialize Spherical K-means

In [56]:
def run_skm(dimension, ncluster, X, niter=20):
    skm = faiss.Kmeans(
        d=dimension, 
        k=ncluster, 
        niter=niter, 
        verbose=True, 
        spherical=True
        )
    skm.train(X)

    centroids = skm.centroids
    _, labels = skm.index.search(X, 1)

    return centroids, labels.flatten()

### Run Experiments Scenario

In [57]:
min_clusters = 2
max_clusters = 10
k_range = range(min_clusters, max_clusters + 1)

results = []
scenario_count = 0

total_iterations = sum(k1 * len(k_range) for k1 in k_range)

with tqdm(total=total_iterations, desc="Evaluasi Hierarchical Clustering") as pbar:
    
    # Loop clustering for first level (k1)
    for k1 in k_range:
        centroids_l1, labels_l1 = run_skm(embedding_dimension, k1, X_embeddings)
        silhouette_avg_l1 = silhouette_score(X_embeddings, labels_l1, metric='cosine')

        for cluster_id in range(k1):
            cluster_indices = np.where(labels_l1 == cluster_id)[0]
            cluster_embeddings = X_embeddings[cluster_indices]

            # Loop clustering for second level (k2)
            for k2 in k_range:
                if len(cluster_embeddings) >= k2:
                    centroids_l2, labels_l2 = run_skm(embedding_dimension, k2, cluster_embeddings)
                    silhouette_avg_l2 = silhouette_score(cluster_embeddings, labels_l2, metric='cosine')

                    results.append({
                        'scenario_id': f"{k1-1}.{cluster_id+1}.{k2-1}",
                        'cluster_id': f"{k1}.{cluster_id+1}",
                        'k1': k1,
                        'k2': k2,
                        'silhouette_avg_l1': f"{silhouette_avg_l1:.4f}",
                        'silhouette_avg_l2': f"{silhouette_avg_l2:.4f}"
                    })
                    scenario_count += 1
                
                pbar.update(1)

print(f"Total scenario yang berhasil dievaluasi: {scenario_count}")

Evaluasi Hierarchical Clustering:   0%|          | 0/486 [00:00<?, ?it/s]

Evaluasi Hierarchical Clustering: 100%|██████████| 486/486 [00:12<00:00, 40.39it/s]

Total scenario yang berhasil dievaluasi: 486


In [58]:
file_path = 'results/experiments/results.csv'

if results:
    fieldnames = list(results[0].keys())
    
    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(results)
    
    print(f"Results saved to {file_path}")

Results saved to results/experiments/results.csv
